# Calculate et0 from BARRA input

In [2]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from dask.distributed import Client
import dask
import tempfile
from et0_barra import calculate_daily_et0

### Only for development phase ###
from et0_barra.et0 import *
from et0_barra.et0 import satVapourPressure
from et0_barra.et0 import meanVar
from et0_barra.et0 import wind2m
from et0_barra.et0 import netRadiation
from et0_barra.et0 import extraTerrRadiation
##################################

# Dask: reuse an existing client if one is already running.
dask.config.set({
    'array.chunk-size': '256 MiB',
    'array.slicing.split_large_chunks': True,
    'distributed.comm.timeouts.connect': '120s',
    'distributed.comm.timeouts.tcp': '120s',
    'distributed.comm.retry.count': 10,
    'distributed.scheduler.allowed-failures': 20,
    'distributed.scheduler.worker-saturation': 1.1,
})
try:
    client
except NameError:
    client = Client(n_workers=7, threads_per_worker=1,
                    local_directory=tempfile.mkdtemp(), memory_limit='32000mb')
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 7,Total memory: 208.62 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38219,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: 2 minutes ago,Total memory: 0 B
Comm: tcp://127.0.0.1:46311,Total threads: 1
Dashboard: /proxy/36517/status,Memory: 29.80 GiB
Nanny: tcp://127.0.0.1:44407,


In [2]:
barra_r2_dir = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/{}/latest/"

In [19]:
# Helper functions
def open_barra2_var(var, file_string=None, verbose=False):
    if var == "orog":
        barra_r2_dir = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/fx/{}/latest/"
        files = glob.glob(f"{barra_r2_dir.format(var)}orog_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1.nc")
    else:
        barra_r2_dir = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/{}/latest/"
        files = sorted(glob.glob(f"{barra_r2_dir.format(var)}*{file_string}*.nc"))
    if verbose:
        for file in files:
            print(file)
    if len(files) > 1:
        return xr.open_mfdataset(files, parallel=True).chunk({"time":3, "lat":228, "lon":383})
    else:
        if var == "orog":
            return xr.open_dataset(files[0]).chunk({"lat":228, "lon":383})
        else:
            return xr.open_dataset(files[0]).chunk({"time":3, "lat":228, "lon":383})

In [20]:
tasmax = open_barra2_var("tasmax","202501")
tasmin = open_barra2_var("tasmin","202501")
rsds = open_barra2_var("rsds","202501")
hurs = open_barra2_var("hurs","202501")
ps = open_barra2_var("ps","202501")
windspeed = open_barra2_var("sfcWind","202501")
orog = open_barra2_var("orog")


In [22]:
orog

<xarray.Dataset> Size: 3MB
Dimensions:  (lat: 646, lon: 1082)
Coordinates:
  * lat      (lat) float64 5kB -57.97 -57.86 -57.75 -57.64 ... 12.76 12.87 12.98
  * lon      (lon) float64 9kB 88.48 88.59 88.7 88.81 ... 207.2 207.3 207.4
    crs      int32 4B ...
Data variables:
    orog     (lat, lon) float32 3MB dask.array<chunksize=(228, 383), meta=np.ndarray>
Attributes: (12/55)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-fx.json
    productive_version:        d031e43
    variable_version:          v20231001
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    date_modified:             2024-10-11T01:52:11Z
    date_metadata_modified:    2024-10-11T01:52:11Z
    history:                   Wed May 08 04:17:21 2024: /g/data/access/ngm/m...
    references:                https://doi.org/10.25914/1x6g-2v48
    license:                   https://doi.org/10.25914/1x6g-2v48
    acknowledgement:           The production of BARRA2 was supported with fu...